# 데이터 셋업 노트북 (최초 1회)

Google 권장 방식: **Drive ↔ Colab = 단일 tar.gz** 전송

```
원본 DB 다운로드 → /content/ (로컬, Drive 거치지 않음)
      ↓
 합성 파이프라인
      ↓
processed/ → tar.gz 단일 압축 → Drive 저장
```

> **최초 1회만 실행** — 이후 세션은 `train_colab.ipynb` 사용

---
## Step 0. 공통 설정 (항상 가장 먼저 실행)

In [ ]:
import os, sys, subprocess

# ── 레포 경로 (변경 시 여기만 수정) ──
REPO_URL = 'https://github.com/sungmin-park-dev/tactical-speech-enhancement.git'
REPO_DIR = '/content/tactical-speech-enhancement'

# 클론 / 업데이트
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)

# sys.path 등록
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'REPO_DIR: {REPO_DIR}')
print(f'sys.path[0]: {sys.path[0]}')
!ls {REPO_DIR}/data/

In [ ]:
%%capture
!pip install soundfile librosa pesq pystoi pyyaml tqdm

In [ ]:
# 경로가 확실히 등록됐는지 확인
import sys
REPO_DIR = '/content/tactical-speech-enhancement'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 모듈 임포트 테스트
from data.noise_mixer import scan_audio_files, NoiseMixer
from data.pipeline import DatasetBuilder, split_by_speaker
from data.bc_simulator import BCSimulator
from data.saturation import SaturationSimulator
print('임포트 성공!')

---
## Step 1. 원본 DB 다운로드 → /content/ 로컬

In [ ]:
import os
!mkdir -p /content/data/raw

# ── LibriSpeech (~6.3GB 압축 / ~11GB 해제) ──
if not os.path.exists('/content/data/raw/LibriSpeech/train-clean-100'):
    print('LibriSpeech 다운로드... (~10분)')
    !cd /content/data/raw && \
        wget -q --show-progress \
        https://www.openslr.org/resources/12/train-clean-100.tar.gz && \
        tar -xzf train-clean-100.tar.gz && \
        rm train-clean-100.tar.gz
    !find /content/data/raw/LibriSpeech -name '*.flac' | wc -l | xargs echo 'flac:'
else:
    !find /content/data/raw/LibriSpeech -name '*.flac' | wc -l | xargs echo '이미 있음. flac:'

In [ ]:
import os

# ── MUSAN (~1.2GB) ──
if not os.path.exists('/content/data/raw/MUSAN/noise'):
    print('MUSAN 다운로드... (~2분)')
    !cd /content/data/raw && \
        wget -q --show-progress \
        http://www.openslr.org/resources/17/musan.tar.gz && \
        tar -xzf musan.tar.gz && mv musan MUSAN && rm musan.tar.gz
    !find /content/data/raw/MUSAN/noise -name '*.wav' | wc -l | xargs echo 'MUSAN noise:'
else:
    !find /content/data/raw/MUSAN/noise -name '*.wav' | wc -l | xargs echo '이미 있음. noise:'

In [ ]:
import os

# ── DEMAND 4개 환경 (~400MB) ──
!mkdir -p /content/data/raw/DEMAND
DEMAND_ENVS = {
    'DKITCHEN': 'https://zenodo.org/record/1227121/files/DKITCHEN_16k.zip',
    'OOFFICE':  'https://zenodo.org/record/1227121/files/OOFFICE_16k.zip',
    'TBUS':     'https://zenodo.org/record/1227121/files/TBUS_16k.zip',
    'TCAR':     'https://zenodo.org/record/1227121/files/TCAR_16k.zip',
}
for name, url in DEMAND_ENVS.items():
    if os.path.exists(f'/content/data/raw/DEMAND/{name}'):
        print(f'  {name} 이미 있음')
        continue
    print(f'  {name} 다운로드...')
    !wget -q --show-progress {url} -O /content/{name}.zip
    !unzip -q /content/{name}.zip -d /content/data/raw/DEMAND/ && rm /content/{name}.zip
    print(f'  {name} 완료')

!find /content/data/raw/DEMAND -name '*.wav' | wc -l | xargs echo 'DEMAND wav:'
!df -h /content | tail -1

---
## Step 2. Colab config 생성

In [ ]:
import sys, yaml
REPO_DIR = '/content/tactical-speech-enhancement'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

with open(f'{REPO_DIR}/configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths'].update({
    'librispeech_root': '/content/data/raw/LibriSpeech',
    'demand_root':      '/content/data/raw/DEMAND',
    'musan_root':       '/content/data/raw/MUSAN',
    'output_root':      '/content/data/processed',
})
COLAB_CFG = '/content/data_config_colab.yaml'
with open(COLAB_CFG, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('Colab config 저장:', COLAB_CFG)

---
## Step 3. 합성 데이터셋 생성

In [ ]:
import sys, numpy as np
REPO_DIR = '/content/tactical-speech-enhancement'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from data.noise_mixer import scan_audio_files
from data.pipeline import DatasetBuilder, split_by_speaker

all_files = scan_audio_files(
    '/content/data/raw/LibriSpeech', extensions=('.flac', '.wav')
)
print(f'LibriSpeech: {len(all_files)}개 파일')
split_files = split_by_speaker(all_files, train_ratio=0.70, val_ratio=0.15, seed=42)

In [ ]:
import sys, numpy as np
REPO_DIR = '/content/tactical-speech-enhancement'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
COLAB_CFG = '/content/data_config_colab.yaml'

from data.pipeline import DatasetBuilder

N_TRAIN, N_VAL, N_TEST = 5000, 1000, 1000  # 빠른 테스트: 500, 100, 100

for env in ['military', 'general']:
    for split, n in [('train', N_TRAIN), ('val', N_VAL), ('test', N_TEST)]:
        print(f'\n=== {env}/{split} ({n}개) ===')
        builder = DatasetBuilder.from_config(
            config_path=COLAB_CFG, env=env,
            speech_files=split_files[split],
            rng=np.random.default_rng(42)
        )
        builder.build(n_samples=n, split=split, verbose=True)

!du -sh /content/data/processed/*
print('\n합성 데이터셋 생성 완료!')

---
## Step 4. Drive에 tar.gz 단일 파일로 저장

> Google 권장: 단일 아카이브로 Drive 저장 → I/O 할당량 절약

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BACKUP = '/content/drive/MyDrive/tactical-speech-enhancement/data'
!mkdir -p {DRIVE_BACKUP}

DRIVE_ARCHIVE = f'{DRIVE_BACKUP}/processed_data.tar.gz'
LOCAL_ARCHIVE = '/content/processed_data.tar.gz'

if os.path.exists(DRIVE_ARCHIVE):
    print(f'이미 있음: {DRIVE_ARCHIVE}')
    print('덮어쓰려면 다음 셀 실행')
else:
    print('로컬에서 압축 중...')
    !tar -czf {LOCAL_ARCHIVE} -C /content/data processed
    !du -sh {LOCAL_ARCHIVE}
    print('Drive로 복사 중... (단일 파일)')
    !cp {LOCAL_ARCHIVE} {DRIVE_ARCHIVE}
    !rm {LOCAL_ARCHIVE}
    !ls -lh {DRIVE_BACKUP}/
    print('\n✅ 완료! 이후 세션: train_colab.ipynb 사용')